In [2]:
!pip install Sastrawi


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 3.6 MB/s eta 0:00:00


In [3]:
import re
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Inisialisasi Sastrawi
stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

stopword_factory = StopWordRemoverFactory()
stopword_remover = stopword_factory.create_stop_word_remover()

# 1. Fungsi Preprocessing Teks
def preprocess_text(text):
    # Hitung jumlah token sebelum preprocessing (berdasarkan spasi)
    token_sebelum = len(text.split())

    # a. Case folding
    text = text.lower()

    # b. Cleaning (menghapus angka, tanda baca, karakter khusus)
    text = re.sub(r'[^a-z\s]', '', text)

    # c. Stopwords removal menggunakan Sastrawi
    text = stopword_remover.remove(text)

    # d. Stemming menggunakan Sastrawi
    text = stemmer.stem(text)

    # e. Tokenisasi
    tokens = text.split()

    return tokens, token_sebelum

# 2. Dataset Minimal 5 Dokumen Berbahasa Indonesia (Tema Pendidikan/Teknologi)
dokumen_list = [
    "Pengumuman: Mahasiswa Fakultas Teknik UNM wajib mengumpulkan laporan praktikum paling lambat tanggal 15 Oktober 2023! Harap perhatikan format penulisan.",
    "Teknologi kecerdasan buatan (AI) saat ini berkembang sangat pesat. Banyak perusahaan teknologi besar berlomba-lomba menciptakan inovasi baru di tahun 2024.",
    "Sistem temu kembali informasi membantu pengguna menemukan dokumen yang relevan dari sekumpulan data yang besar dan tidak terstruktur.",
    "Pendaftaran beasiswa untuk mahasiswa berprestasi telah dibuka. Persyaratan lengkap dapat dilihat pada papan pengumuman di gedung dekanat Fakultas.",
    "Workshop pengembangan aplikasi mobile menggunakan framework Flutter akan dilaksanakan hari Sabtu besok. Kuota terbatas hanya untuk 50 peserta!"
]

# Variabel untuk menampung statistik
total_token_sebelum = 0
total_token_sesudah = 0

print("=== HASIL PREPROCESSING TEKS ===\n")

# 3 & 4. Terapkan fungsi, tampilkan perbandingan, dan hitung token
for i, teks_asli in enumerate(dokumen_list):
    tokens_sesudah, jml_sebelum = preprocess_text(teks_asli)
    jml_sesudah = len(tokens_sesudah)

    total_token_sebelum += jml_sebelum
    total_token_sesudah += jml_sesudah

    # Menampilkan perbandingan secara lengkap (semua dokumen ditampilkan agar representatif)
    print(f"Dokumen ke-{i+1}")
    print(f"Sebelum : {teks_asli}")
    print(f"Sesudah : {tokens_sesudah}")
    print(f"Token Sebelum : {jml_sebelum}")
    print(f"Token Sesudah : {jml_sesudah}")
    print("-" * 50)

# Menghitung Persentase Pengurangan Token
pengurangan_token = total_token_sebelum - total_token_sesudah
persentase_pengurangan = (pengurangan_token / total_token_sebelum) * 100 if total_token_sebelum > 0 else 0

print("\n=== RINGKASAN STATISTIK TOKEN ===")
print(f"Total Token Sebelum Preprocessing : {total_token_sebelum}")
print(f"Total Token Setelah Preprocessing : {total_token_sesudah}")
print(f"Persentase Pengurangan Token    : {persentase_pengurangan:.2f}%")

=== HASIL PREPROCESSING TEKS ===

Dokumen ke-1
Sebelum : Pengumuman: Mahasiswa Fakultas Teknik UNM wajib mengumpulkan laporan praktikum paling lambat tanggal 15 Oktober 2023! Harap perhatikan format penulisan.
Sesudah : ['umum', 'mahasiswa', 'fakultas', 'teknik', 'unm', 'wajib', 'kumpul', 'lapor', 'praktikum', 'paling', 'lambat', 'tanggal', 'oktober', 'harap', 'perhati', 'format', 'tulis']
Token Sebelum : 19
Token Sesudah : 17
--------------------------------------------------
Dokumen ke-2
Sebelum : Teknologi kecerdasan buatan (AI) saat ini berkembang sangat pesat. Banyak perusahaan teknologi besar berlomba-lomba menciptakan inovasi baru di tahun 2024.
Sesudah : ['teknologi', 'cerdas', 'buat', 'ai', 'ini', 'kembang', 'sangat', 'pesat', 'banyak', 'usaha', 'teknologi', 'besar', 'berlombalomba', 'cipta', 'inovasi', 'baru', 'tahun']
Token Sebelum : 20
Token Sesudah : 17
--------------------------------------------------
Dokumen ke-3
Sebelum : Sistem temu kembali informasi membantu pengguna

Preprocessing meningkatkan kualitas data IR secara signifikan dengan menghapus noise (seperti tanda baca dan kata hubung) serta menormalisasi kata ke bentuk dasarnya melalui stemming. Proses ini secara efektif mereduksi ukuran indeks dan membuang kata yang tidak informatif, sehingga pencocokan kueri pencarian menjadi jauh lebih cepat, hemat memori, dan presisi.

In [4]:
import re
import math
import pandas as pd
import numpy as np
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from sklearn.feature_extraction.text import TfidfVectorizer

# Dataset (4 Dokumen Topik Komputer & Teknologi)
dokumen_list = [
    "Sistem komputer terdiri dari perangkat keras dan perangkat lunak.",
    "Jaringan komputer menghubungkan berbagai perangkat untuk berbagi data.",
    "Kecerdasan buatan membantu komputer menganalisis data secara otomatis.",
    "Sistem temu kembali membantu pencarian dokumen berbasis teks."
]

# 1. Preprocessing Sederhana (Case Folding + Tokenisasi + Stopwords Removal)
stopword_factory = StopWordRemoverFactory()
stopword_remover = stopword_factory.create_stop_word_remover()

def preprocess(text):
    text = text.lower() # Case folding
    text = re.sub(r'[^a-z\s]', '', text) # Cleaning
    text = stopword_remover.remove(text) # Stopwords removal
    return text.strip()

corpus_clean = [preprocess(doc) for doc in dokumen_list]

# Kosakata Unik (Vocabulary)
vocab = sorted(list(set(" ".join(corpus_clean).split())))
doc_labels = [f"Doc {i+1}" for i in range(len(dokumen_list))]

# 2. Bag-of-Words (Raw Count)
bow_matrix = []
for doc in corpus_clean:
    tokens = doc.split()
    bow_matrix.append([tokens.count(term) for term in vocab])

df_bow = pd.DataFrame(bow_matrix, columns=vocab, index=doc_labels)

# 3. Perhitungan Manual (TF, DF, IDF, TF-IDF)
df_tf = df_bow.copy() # TF = Raw Count

# DF & IDF Manual (IDF = log10(N / DF))
N = len(dokumen_list)
df_count = (df_bow > 0).sum(axis=0)
idf_manual = df_count.apply(lambda df_val: math.log10(N / df_val))

# TF-IDF Manual = TF * IDF
df_tfidf_manual = df_bow.copy().astype(float)
for term in vocab:
    df_tfidf_manual[term] = df_tf[term] * idf_manual[term]

# 4 & 5. Implementasi scikit-learn (TfidfVectorizer) & Matriks TF-IDF
vectorizer = TfidfVectorizer()
tfidf_sklearn = vectorizer.fit_transform(corpus_clean)
df_tfidf_sklearn = pd.DataFrame(tfidf_sklearn.toarray(), columns=vectorizer.get_feature_names_out(), index=doc_labels)

# Menampilkan Hasil Kebagian Output Notebook
print("=== 1. CORPUS SETELAH PREPROCESSING ===")
for i, doc in enumerate(corpus_clean):
    print(f"Doc {i+1}: {doc}")

print("\n=== 2. MATRIKS BAG-OF-WORDS (RAW COUNT) ===")
print(df_bow)

print("\n=== 3. DOCUMENT FREQUENCY (DF) & IDF MANUAL ===")
df_idf_summary = pd.DataFrame({"DF": df_count, "IDF Manual": idf_manual.round(4)})
print(df_idf_summary.T)

print("\n=== 5a. MATRIKS TF-IDF (MANUAL) ===")
print(df_tfidf_manual.round(4))

print("\n=== 5b. MATRIKS TF-IDF (SCIKIT-LEARN) ===")
print(df_tfidf_sklearn.round(4))

=== 1. CORPUS SETELAH PREPROCESSING ===
Doc 1: sistem komputer terdiri perangkat keras perangkat lunak
Doc 2: jaringan komputer menghubungkan berbagai perangkat berbagi data
Doc 3: kecerdasan buatan membantu komputer menganalisis data otomatis
Doc 4: sistem temu membantu pencarian dokumen berbasis teks

=== 2. MATRIKS BAG-OF-WORDS (RAW COUNT) ===
       berbagai  berbagi  berbasis  buatan  data  dokumen  jaringan  \
Doc 1         0        0         0       0     0        0         0   
Doc 2         1        1         0       0     1        0         1   
Doc 3         0        0         0       1     1        0         0   
Doc 4         0        0         1       0     0        1         0   

       kecerdasan  keras  komputer  ...  membantu  menganalisis  \
Doc 1           0      1         1  ...         0             0   
Doc 2           0      0         1  ...         0             0   
Doc 3           1      0         1  ...         1             1   
Doc 4           0      0   

Kata-kata dengan bobot TF-IDF tertinggi di tiap dokumen adalah kata-kata khusus yang hanya muncul pada dokumen tersebut, seperti kata keras dan lunak di Dokumen 1, jaringan di Dokumen 2, kecerdasan di Dokumen 3, serta pencarian dan dokumen di Dokumen 4. Kata-kata spesifik ini sangat penting karena berfungsi sebagai identitas utama yang membedakan satu dokumen dari dokumen lainnya. Sementara itu, kata umum yang muncul di hampir semua dokumen (seperti kata komputer) akan mendapatkan bobot yang rendah karena tidak membantu sistem dalam mengenali keunikan isi dokumen tersebut.